# Jorg GPU Acceleration Tutorial

This notebook demonstrates how to use GPU acceleration in Jorg for faster stellar spectral synthesis.

## Requirements
- NVIDIA GPU with CUDA support
- JAX with CUDA: `pip install jax[cuda] jaxlib[cuda]`

If you don't have a GPU, the code will automatically fall back to CPU mode.

## 1. Check GPU Availability

In [ ]:
import os
import sys
from pathlib import Path

# Add src to path for editable installs (optional)
repo_root = Path.cwd()
while repo_root != repo_root.parent:
    src_path = repo_root / "src"
    if (src_path / "jorg").exists():
        sys.path.insert(0, str(src_path))
        data_dir = repo_root / "data"
        if data_dir.exists():
            os.environ.setdefault("JORG_DATA_DIR", str(data_dir))
        break
    repo_root = repo_root.parent

# Initialize JAX and check device
from jorg.gpu import init_jax, get_device_info, is_gpu_available


## 2. Basic Synthesis (GPU-accelerated)

The API is unchanged - GPU acceleration happens automatically when available.

In [ ]:
from jorg.synthesis import synth
from jorg.lines.linelist_data import get_VALD_solar_linelist
import numpy as np
import matplotlib.pyplot as plt

# Solar spectrum synthesis
wl, flux, cont = synth(
    Teff=5780,
    logg=4.44,
    m_H=0.0,
    wavelengths=(5000, 5020),  # Small range for quick demo
    linelist=get_VALD_solar_linelist(),
    verbose=True
)

# Plot
plt.figure(figsize=(10, 4))
plt.plot(wl, flux, 'k-', linewidth=0.5)
plt.xlabel('Wavelength (Å)')
plt.ylabel('Normalized Flux')
plt.title('Solar Spectrum (GPU-accelerated)')
plt.grid(alpha=0.3)
plt.show()

## 3. Performance Benchmark (GPU if available)

We'll time a few synthesis runs. If a CUDA GPU is available, JAX will use it automatically; otherwise it will run on CPU.


In [ ]:
from jorg.synthesis import synthesize_korg_compatible, create_korg_compatible_abundance_array
from jorg.atmosphere import interpolate_marcs
from jorg.gpu import timed_block
import time

# Setup test case (metal-poor dwarf)
Teff = 4000
logg = 4.5
m_H = -2.0

# Wavelength window for benchmarks
wl_range = (5000, 5020)

print(f"Test case: Teff={Teff}K, logg={logg}, [M/H]={m_H}")
print("Setting up atmosphere...")

atm = interpolate_marcs(Teff, logg, m_H)
A_X = create_korg_compatible_abundance_array(m_H)

# Convert atmosphere to dict
if hasattr(atm, 'layers'):
    atm_dict = {
        'temperature': np.array([layer.temp for layer in atm.layers]),
        'electron_density': np.array([layer.electron_number_density for layer in atm.layers]),
        'number_density': np.array([layer.number_density for layer in atm.layers]),
        'tau_5000': np.array([layer.tau_5000 for layer in atm.layers]),
        'height': np.array([layer.z for layer in atm.layers]),
    }
    atm_dict['pressure'] = atm_dict['number_density'] * 1.380649e-16 * atm_dict['temperature']
else:
    atm_dict = atm

n_layers = len(atm_dict['temperature'])
print(f"Atmosphere has {n_layers} layers")

### Warm-up run (JIT compilation)

In [ ]:
print("Warm-up run (JAX compile where applicable)...")
print("This may take a few seconds...\n")

with timed_block("JIT compilation", verbose=True):
    result_warmup = synthesize_korg_compatible(
        atm_dict,
        [],  # Continuum only
        A_X,
        wavelengths=wl_range,
        hydrogen_lines=False,
        verbose=False,
        logg=logg,
    )

print("\nWarm-up complete! Subsequent runs will be faster.")


### Timed benchmark (post-JIT)

In [ ]:
def run_benchmark(label, runs=3):
    times = []
    last_result = None
    for i in range(runs):
        with timed_block(f"{label} Run {i+1}", verbose=True) as t:
            last_result = synthesize_korg_compatible(
                atm_dict,
                [],
                A_X,
                wavelengths=wl_range,
                hydrogen_lines=False,
                verbose=False,
                logg=logg,
            )
        times.append(t['elapsed'])
    return float(np.mean(times)), float(np.std(times)), last_result

print("Running benchmark...\n")

label = "JAX/GPU" if is_gpu_available() else "JAX/CPU"
runs = 3
avg, std, result = run_benchmark(label, runs=runs)

print(f"\n{'='*50}")
print(f"{label} average: {avg:.3f}s ± {std:.3f}s (n={runs})")
print(f"{'='*50}")

if is_gpu_available():
    print(f"\n✓ Running on GPU ({device_info['device_kind']})")
else:
    print("\n⚠ Running on CPU")
    print("  Install JAX CUDA for GPU acceleration: pip install jax[cuda]")


## 4. Correctness Verification

Verify that GPU results are accurate.

In [ ]:
flux = np.asarray(result.flux)
cntm = np.asarray(result.cntm) if result.cntm is not None else flux
alpha = np.asarray(result.alpha)

print("Output verification:")
print(f"  Flux shape: {flux.shape}")
print(f"  Alpha shape: {alpha.shape}")
print(f"  Flux range: [{flux.min():.3e}, {flux.max():.3e}]")
print(f"  Alpha range: [{alpha.min():.3e}, {alpha.max():.3e}]")
print()

# Check for numerical issues
checks = [
    ("NaN values", not (np.any(np.isnan(flux)) or np.any(np.isnan(alpha)))),
    ("Inf values", not (np.any(np.isinf(flux)) or np.any(np.isinf(alpha)))),
    ("Positive flux", np.all(flux > 0)),
]

print("Quality checks:")
for check_name, passed in checks:
    symbol = "✓" if passed else "✗"
    print(f"  {symbol} {check_name}")

# Check continuum rectification
if cntm is not None:
    rect = flux / (cntm + 1e-100)
    rect_ok = np.allclose(rect, 1.0, rtol=0.01)
    symbol = "✓" if rect_ok else "~"
    print(f"  {symbol} Continuum rectification: [{rect.min():.6f}, {rect.max():.6f}]")
    print(f"    (Should be ≈1.0 for continuum-only synthesis)")

## 5. Advanced: Direct Chemical Equilibrium Benchmarking

Compare the JAX-accelerated chemical equilibrium solver directly.